In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr,col
from pyspark.sql import functions as F
import time

In [9]:
import os
CATALOG = "lakehouse"
C = f"spark.sql.catalog.{CATALOG}"

spark = (
    SparkSession.builder
    .appName("Lakehouse Implementation")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # Named Iceberg catalog backed by the Hive Metastore
    .config(C, "org.apache.iceberg.spark.SparkCatalog")
    .config(f"{C}.type", "hive")
    .config(f"{C}.uri", "thrift://hive-metastore:9083")
    .config(f"{C}.warehouse", "s3://buck1/warehouse")

    # Iceberg's own S3 client for table data and metadata.
    # Keys and region come from AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_REGION.
    .config(f"{C}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"{C}.s3.endpoint", os.environ["S3_ENDPOINT"])
    .config(f"{C}.s3.path-style-access", "true")
    .config(f"{C}.http-client.type", "apache")   # the only HTTP client in bundle-2.29.52.jar

    .config("spark.sql.defaultCatalog", CATALOG)

    # S3A is still used for streaming checkpoints (Spark writes them through Hadoop, not Iceberg).
    # S3A's default credential chain reads the same AWS_* environment variables.
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .getOrCreate()
)

spark

In [3]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|        lakehouse|
+-----------------+



In [7]:
spark.sql("""
    CREATE DATABASE IF NOT EXISTS demo
""")

DataFrame[]

In [3]:
print(
    spark.sql(
        "SHOW DATABASES"
    ).show()
)

+---------+
|namespace|
+---------+
|  default|
|     demo|
+---------+

None


In [4]:
print(
    spark.sql(
        "SHOW Tables in demo"
    ).show()
)

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|     demo|cache_test|      false|
+---------+----------+-----------+

None


In [9]:
# Create an Iceberg table
spark.sql("""
    CREATE TABLE demo.cache_test (
        id BIGINT,
        name STRING,
        category STRING,
        value DOUBLE,
        created_at TIMESTAMP
    )
    USING iceberg
""")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


DataFrame[]

In [7]:
df_write = (
    spark.range(0, 1_000_000)
    .withColumn("name", F.concat(F.lit("user_"), F.col("id")))
    .withColumn(
        "category",
        F.when((F.col("id") % 3) == 0, "A")
         .when((F.col("id") % 3) == 1, "B")
         .otherwise("C")
    )
    .withColumn("value", (F.rand() * 1000))
    .withColumn(
        "created_at",
        F.expr("timestamp('2026-01-01') + INTERVAL 1 SECOND * id")
    )
    .select("id", "name", "category", "value", "created_at")
)
df_write.writeTo("demo.cache_test").append()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [5]:
spark.sql("""
    SELECT COUNT(*)
    FROM demo.cache_test
""").show()

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+--------+
|count(1)|
+--------+
| 1000000|
+--------+



In [6]:
df=spark.sql(
    """
    SELECT
        id,
        name,
        category,
        value,
        created_at
    FROM demo.cache_test
    WHERE value < 300
    """
)

## Test JVM side functionalities of Spark

In [9]:
# Calling JVM side spark APIs from python through Py4J instead of normal Python-facing APIs
qe = df._jdf.queryExecution()
plan=qe.optimizedPlan()
print(plan.toString())

Filter (isnotnull(value#47) AND (value#47 < 300.0))
+- RelationV2[id#44L, name#45, category#46, value#47, created_at#48] lakehouse.demo.cache_test



In [8]:
print(qe.analyzed().toString())      # analyzed logical plan
print(qe.executedPlan().toString())  # physical plan

Project [id#44L, name#45, category#46, value#47, created_at#48]
+- Filter (value#47 < cast(300 as double))
   +- SubqueryAlias lakehouse.demo.cache_test
      +- RelationV2[id#44L, name#45, category#46, value#47, created_at#48] lakehouse.demo.cache_test

*(1) Filter (isnotnull(value#47) AND (value#47 < 300.0))
+- *(1) ColumnarToRow
   +- BatchScan lakehouse.demo.cache_test[id#44L, name#45, category#46, value#47, created_at#48] IcebergScan(table=lakehouse.demo.cache_test, schemaId=0, snapshotId=7843675569698251131, branch=null, filters=value IS NOT NULL, value < 300.0, runtimeFilters=, groupedBy=) RuntimeFilters: []



In [11]:
estimated_bytes = int(plan.stats().sizeInBytes())
estimated_bytes  # is intended to represent the estimated size of the output produced by that plan

64000000

In [14]:
def _find_in_memory_relation(plan, max_nodes: int = 64):
    """Breadth-first hunt for the `InMemoryRelation` in a Catalyst plan, or None.

    The cached relation is usually the whole optimized plan, but a projection or filter can sit on
    top of it. Walked by hand because `plan.collect` takes a Scala partial function, which py4j
    cannot construct.
    """
    queue = [plan]
    seen = 0
    while queue and seen < max_nodes:
        node = queue.pop(0)
        seen += 1
        if node.getClass().getSimpleName() == "InMemoryRelation":
            return node
        children = node.children()
        queue.extend(children.apply(i) for i in range(children.size()))
    return None
relation = _find_in_memory_relation(plan)  # This is related to the caching, If the DataFrame is represented as cached, you may find InMemoryRelation
print(relation)

None


In [28]:
def _cached_rdd_id(relation) -> int | None:
    """Id of the RDD holding `relation`'s cached batches, or None if no accessor resolves.

    `InMemoryRelation.cachedColumnBuffers` is not reachable over py4j on every build (EMR's
    3.5.6-amzn-2 answers `Method cachedColumnBuffers([]) does not exist`), so go through the
    `cacheBuilder` field first and keep the direct call as the fallback.
    """
    for path in (
        lambda: relation.cacheBuilder().cachedColumnBuffers().id(),
        lambda: relation.cachedColumnBuffers().id(),
    ):
        try:
            return int(path())
        except Exception as e:
            print(f"Cached-RDD accessor not available on this build: {type(e).__name__}: {e}")
            continue

    try:
        methods = sorted({str(m.getName()) for m in relation.getClass().getMethods() if "ache" in str(m.getName())})
        print(f"No usable cached-RDD accessor on InMemoryRelation. Cache-ish methods: {methods}")
    except Exception as e:
        print(f"No usable cached-RDD accessor on InMemoryRelation, and could not list its methods: {e}")
    return None

In [29]:
rdd_id = _cached_rdd_id(relation)
cached_rdds = list(spark.sparkContext._jsc.sc().getRDDStorageInfo())  # type: ignore[attr-defined]
for info in cached_rdds:
    if rdd_id is not None and int(info.id()) == rdd_id:
        memory_bytes = int(info.memSize())
        disk_bytes = int(info.diskSize())
        cached_partitions = int(info.numCachedPartitions())
        partitions = int(info.numPartitions())

Cached-RDD accessor not available on this build: AttributeError: 'NoneType' object has no attribute 'cacheBuilder'
Cached-RDD accessor not available on this build: AttributeError: 'NoneType' object has no attribute 'cachedColumnBuffers'
No usable cached-RDD accessor on InMemoryRelation, and could not list its methods: 'NoneType' object has no attribute 'getClass'


26/10/05 07:45:47 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 473163 ms exceeds timeout 120000 ms
26/10/05 07:45:47 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 474432 ms exceeds timeout 120000 ms
26/10/05 07:45:48 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:748)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:104)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:216)
	at org.apache.spark.rpc.netty.Inbox.process(Inbox.scala:101)
	at org.apache.spark.rpc.netty.MessageLoop.org$apache$s

In [30]:
df_cached = df.cache()

# This materializes/fills the cache
df_cached.count()

26/10/05 07:59:18 WARN CacheManager: Asked to cache already cached data.


300151

## Test the table caching and verifiied from the SQL/DataFrame Dag

In [30]:
df=spark.sql("""
    SELECT *
    FROM demo.cache_test
""")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


In [32]:
df_cached = df.cache()

# This materializes/fills the cache
df_cached.count()

1000000

In [29]:
df.unpersist()

DataFrame[id: bigint, name: string, category: bigint]

## Manual Broadcasting

In [10]:
from pyspark.sql import functions as F

small_df = spark.range(1000).withColumn(
    "rule_id", F.col("id")
)

small_df.show(5)

+---+-------+
| id|rule_id|
+---+-------+
|  0|      0|
|  1|      1|
|  2|      2|
|  3|      3|
|  4|      4|
+---+-------+
only showing top 5 rows


In [11]:
broadcast_df = F.broadcast(small_df)

broadcast_df.explain("formatted")

== Physical Plan ==
* Project (2)
+- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#18L]
Arguments: Range (0, 1000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [id#18L, id#18L AS rule_id#19L]
Input [1]: [id#18L]




26/10/06 11:02:34 WARN HintErrorLogger: A join hint (strategy=broadcast) is specified but it is not part of a join relation.


In [12]:
large_df = spark.range(1_000_000).withColumn(
    "rule_id",
    (F.col("id") % 1000)
)

In [13]:
result = large_df.join(
    broadcast_df,
    "rule_id"
)

result.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- BroadcastHashJoin Inner BuildRight (7)
      :- Project (3)
      :  +- Filter (2)
      :     +- Range (1)
      +- BroadcastExchange (6)
         +- Project (5)
            +- Range (4)


(1) Range
Output [1]: [id#27L]
Arguments: Range (0, 1000000, step=1, splits=Some(4))

(2) Filter
Input [1]: [id#27L]
Condition : isnotnull((id#27L % 1000))

(3) Project
Output [2]: [id#27L, (id#27L % 1000) AS rule_id#28L]
Input [1]: [id#27L]

(4) Range
Output [1]: [id#18L]
Arguments: Range (0, 1000, step=1, splits=Some(2))

(5) Project
Output [2]: [id#18L, id#18L AS rule_id#19L]
Input [1]: [id#18L]

(6) BroadcastExchange
Input [2]: [id#18L, rule_id#19L]
Arguments: HashedRelationBroadcastMode(List(input[1, bigint, false]),false), [plan_id=216]

(7) BroadcastHashJoin
Left keys [1]: [rule_id#28L]
Right keys [1]: [rule_id#19L]
Join type: Inner
Join condition: None

(8) Project
Output [3]: [rule_id#28L, id#27L, id#18L]
Input [4]: [id#27L, rul

In [14]:
result.count()

1000000

26/10/06 11:44:40 WARN HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 985310 ms exceeds timeout 120000 ms
26/10/06 11:44:40 WARN HeartbeatReceiver: Removing executor 1 with no recent heartbeats: 991577 ms exceeds timeout 120000 ms
26/10/06 11:44:40 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.10: worker lost: Not receiving heartbeat for 60 seconds
26/10/06 11:44:40 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.11: worker lost: Not receiving heartbeat for 60 seconds
26/10/06 11:44:40 ERROR Inbox: Ignoring error
java.lang.AssertionError: assertion failed: BlockManager re-registration shouldn't succeed when the executor is lost
	at scala.Predef$.assert(Predef.scala:279)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$register(BlockManagerMasterEndpoint.scala:748)
	at org.apache.spark.storage.BlockManagerMasterEndpoint$$anonfun$receiveAndReply$1.applyOrElse(BlockManagerMasterEndpoint.scala:141)
	

In [8]:
spark.stop()

In [15]:
spark_executor_instances = spark.conf.get("spark.executor.instances", None)
spark_executor_cores = spark.conf.get("spark.executor.cores", None)